# SALARIES OLD

In [83]:
from pathlib import Path
import os

PROJECT_DIR = Path(r"C:\Users\marco\OneDrive\Desktop\Proyecto")
os.chdir(PROJECT_DIR)

print("Directorio de trabajo:", Path.cwd())

Directorio de trabajo: C:\Users\marco\OneDrive\Desktop\Proyecto


In [84]:
import pandas as pd
import numpy as np

In [85]:
salary_old = pd.read_csv(r"Datasets\Salaries - Kaggle\NBA Salaries(1990-2023).csv")

print(salary_old.shape)

print(salary_old.columns)

print(salary_old.head())

(15857, 5)
Index(['Unnamed: 0', 'playerName', 'seasonStartYear', 'salary',
       'inflationAdjSalary'],
      dtype='object')
   Unnamed: 0        playerName  seasonStartYear      salary  \
0           0     Patrick Ewing             1990  $4,250,000   
1           1  Hot Rod Williams             1990  $3,785,000   
2           2   Hakeem Olajuwon             1990  $3,175,000   
3           3   Charles Barkley             1990  $2,900,000   
4           4      Chris Mullin             1990  $2,850,000   

  inflationAdjSalary  
0         $9,694,547  
1         $8,633,850  
2         $7,242,397  
3         $6,615,103  
4         $6,501,049  


In [86]:
print(salary_old.dtypes)

Unnamed: 0             int64
playerName            object
seasonStartYear        int64
salary                object
inflationAdjSalary    object
dtype: object


## Limpieza

### 1. Eliminar la columna índice

In [87]:
salary_old = salary_old.drop(columns="Unnamed: 0")

### 2. Renombrar columnas

In [88]:
salary_old = salary_old.rename(
    columns={
        "playerName": "Player",
        "seasonStartYear": "SeasonYear",
        "salary": "Salary",
        "inflationAdjSalary": "SalaryInflation"
    }
)

### 3. Crear la columna Season

In [89]:
salary_old["Season"] = (
    salary_old["SeasonYear"].astype(str)
    + "-"
    + (salary_old["SeasonYear"] + 1).astype(str).str[-2:]
)

### 4. Convertir salarios a numéricos

In [90]:
salary_old["Salary"] = (
    salary_old["Salary"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

salary_old["SalaryInflation"] = (
    salary_old["SalaryInflation"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [91]:
salary_old = salary_old[
    salary_old["SeasonYear"] >= 2000
].copy()

salary_old.reset_index(drop=True, inplace=True)

### 5. Definimos las columnas que usaremos

In [92]:
salary_old = salary_old[
    [
        "Player",
        "Season",
        "Salary",
        "SalaryInflation"
    ]
].copy()

### 5. Auditoría

In [93]:
print(salary_old.shape)

print(salary_old.dtypes)

print(salary_old.head())

print()

print(salary_old["Season"].min())

print(salary_old["Season"].max())

print()

print(salary_old.isna().sum())

(11666, 4)
Player              object
Season              object
Salary             float64
SalaryInflation    float64
dtype: object
             Player   Season      Salary  SalaryInflation
0     Kevin Garnett  2000-01  19610000.0       33704516.0
1  Shaquille O'Neal  2000-01  19285715.0       33147154.0
2   Alonzo Mourning  2000-01  16880000.0       29012353.0
3      Juwan Howard  2000-01  16875000.0       29003759.0
4   Hakeem Olajuwon  2000-01  16700000.0       28702979.0

2000-01
2021-22

Player             0
Season             0
Salary             0
SalaryInflation    0
dtype: int64


In [94]:
print(
    salary_old[
        ["Player", "Season"]
    ].duplicated().sum()
)

654


In [95]:
salary_old[
    ["Player", "Season"]
].value_counts().head(20)

Player             Season 
Fred VanVleet      2021-22    2
Anthony Lamb       2021-22    2
John Wall          2021-22    2
Carlik Jones       2021-22    2
Cody Zeller        2021-22    2
Freddie Gillespie  2021-22    2
Jonathan Isaac     2021-22    2
Damian Lillard     2021-22    2
Jaylen Hoard       2021-22    2
Immanuel Quickley  2021-22    2
Chris Boucher      2021-22    2
Jeff Green         2021-22    2
Ed Davis           2021-22    2
Jahlil Okafor      2021-22    2
DeMar DeRozan      2021-22    2
James Harden       2021-22    2
Furkan Korkmaz     2021-22    2
Cole Anthony       2021-22    2
Draymond Green     2021-22    2
Zylan Cheatham     2021-22    2
Name: count, dtype: int64

In [96]:
salary_old[
    salary_old.duplicated(
        subset=["Player","Season"],
        keep=False
    )
].sort_values(
    ["Player","Season"]
).head(30)

,Player,Season,Salary,SalaryInflation
10437,Aaron Gordon,2021-22,16409091.0,17895714.0
11090,Aaron Gordon,2021-22,16409091.0,17895714.0
10926,Aaron Henry,2021-22,223338.0,243571.0
11579,Aaron Henry,2021-22,223338.0,243571.0
10613,Aaron Holiday,2021-22,3980551.0,4341179.0
11266,Aaron Holiday,2021-22,3980551.0,4341179.0
10625,Aaron Nesmith,2021-22,3631200.0,3960177.0
11278,Aaron Nesmith,2021-22,3631200.0,3960177.0
10811,Aaron Wiggins,2021-22,1305761.0,1424059.0
11464,Aaron Wiggins,2021-22,1305761.0,1424059.0


#### Borramos los duplicados, ya que son filas idénticas

In [97]:
salary_old = salary_old.drop_duplicates(
    subset=["Player", "Season"],
    keep="first"
).reset_index(drop=True)

#### Comprobamos de nuevo

In [98]:
print(salary_old.shape)

print(
    salary_old[
        ["Player", "Season"]
    ].duplicated().sum()
)

(11012, 4)
0


### 6. Auditoría de compatibilidad con nba_master_featured

In [99]:
nba_master_featured = pd.read_csv(r"Datasets\Team Stats - Kaggle\nba_master_featured.csv")

In [100]:
print(sorted(nba_master_featured["Season"].unique()))
print(sorted(salary_old["Season"].unique()))

['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22']


In [101]:
print(nba_master_featured["Season"].min(), nba_master_featured["Season"].max())
print(salary_old["Season"].min(), salary_old["Season"].max())

2000-01 2024-25
2000-01 2021-22


In [102]:
print(nba_master_featured.shape)
print(salary_old.shape)

(12227, 179)
(11012, 4)


- Jugadores en ambos datasets

In [103]:
master_players = set(nba_master_featured["Player"].unique())
salary_players = set(salary_old["Player"].unique())

print("Jugadores master:", len(master_players))
print("Jugadores salary:", len(salary_players))

print("Coinciden:", len(master_players & salary_players))
print("Solo master:", len(master_players - salary_players))
print("Solo salary:", len(salary_players - master_players))

Jugadores master: 2473
Jugadores salary: 2229
Coinciden: 1913
Solo master: 560
Solo salary: 316


In [104]:
test_merge = nba_master_featured.merge(
    salary_old,
    on=["Player", "Season"],
    how="left",
    indicator=True
)

print(test_merge["_merge"].value_counts())

print(test_merge["Salary"].isna().sum())

_merge
both          9291
left_only     2936
right_only       0
Name: count, dtype: int64
2936


In [105]:
print(
    test_merge.loc[
        test_merge["Salary"].isna(),
        "Season"
    ]
    .value_counts()
    .sort_index()
)

Season
2000-01     40
2001-02     39
2002-03     40
2003-04     55
2004-05     60
2005-06     52
2006-07     46
2007-08     44
2008-09     44
2009-10     49
2010-11     52
2011-12     66
2012-13     47
2013-14     61
2014-15     58
2015-16     58
2016-17     66
2017-18     62
2018-19     72
2019-20     97
2020-21     72
2021-22     76
2022-23    539
2023-24    572
2024-25    569
Name: count, dtype: int64


#### Verificamos que se haya corregido realmente el problema con el encode de los nombres

In [106]:
for p in [
    "Peja",
    "Hedo",
    "Nájera",
    "Şengün",
    "Stojaković",
    "Türkoğlu",
    "Varejão"
]:

    print("\n", p)

    print(
        nba_master_featured[
            nba_master_featured["Player"].str.contains(p, na=False)
        ][["Player"]].drop_duplicates()
    )


 Peja
             Player
22  Peja Stojaković

 Hedo
            Player
248  Hedo Türkoğlu

 Nájera
             Player
337  Eduardo Nájera

 Şengün
               Player
10135  Alperen Şengün

 Stojaković
             Player
22  Peja Stojaković

 Türkoğlu
            Player
248  Hedo Türkoğlu

 Varejão
                Player
2040  Anderson Varejão


In [107]:
missing = test_merge[
    (test_merge["Salary"].isna()) &
    (test_merge["Season"] <= "2021-22")
]

missing[
    ["Player", "Season", "Team"]
].drop_duplicates().head(100)

,Player,Season,Team
22,Peja Stojaković,2000-01,SAC
105,Toni Kukoč,2000-01,PHI
154,P.J. Brown,2000-01,CHH
169,Felipe López,2000-01,WAS
223,A.J. Guyton,2000-01,CHI
...,...,...,...
1127,Robert Pack,2002-03,NOH
1136,Jake Tsakalidis,2002-03,PHO
1159,Corey Benjamin,2002-03,ATL
1168,Tierre Brown,2002-03,CLE


### 7. Normalización de nombres

#### 7.1. Crear una clave auxiliar

In [108]:
import re
import unicodedata
import numpy as np

def create_player_key(name):

    if pd.isna(name):
        return np.nan

    # Eliminar acentos
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    # Eliminar puntos y apóstrofes
    name = re.sub(r"[.'`´]", "", name)

    # Eliminar guiones
    name = name.replace("-", " ")

    # Eliminar sufijos
    name = re.sub(
        r"\b(JR|SR|II|III|IV)\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Espacios
    name = " ".join(name.split())

    return name.lower()

In [109]:
nba_master_featured["PlayerKey"] = (
    nba_master_featured["Player"]
    .apply(create_player_key)
)

salary_old["PlayerKey"] = (
    salary_old["Player"]
    .apply(create_player_key)
)

In [110]:
test_merge = nba_master_featured.merge(
    salary_old,
    on=["PlayerKey", "Season"],
    how="left",
    suffixes=("", "_salary"),
    indicator=True
)

In [111]:
print(test_merge["_merge"].value_counts())

print(test_merge["Salary"].isna().sum())

_merge
both          10084
left_only      2143
right_only        0
Name: count, dtype: int64
2143


### Auditoría final

1. Separar los casos "esperados" de los "inesperados".
Primero eliminamos las temporadas que sabemos que no existen en el dataset de salarios.

In [112]:
missing_salary = test_merge[
    (test_merge["Salary"].isna()) &
    (~test_merge["Season"].isin(["2022-23", "2023-24", "2024-25"]))
].copy()

print(missing_salary.shape)

(463, 184)


2. ¿Cuántos jugadores únicos son realmente?

In [113]:
print(
    missing_salary["Player"]
    .nunique()
)

missing_players = (
    missing_salary["Player"]
    .drop_duplicates()
    .sort_values()
)

print(missing_players.head(100))

236
1303         A.J. Guyton
5784          Aaron Gray
6275       Adonis Thomas
5079       Alan Anderson
9244        Allen Crabbe
              ...       
10453      Ish Wainright
3054          J.J. Barea
2104       Jackie Butler
290      Jake Tsakalidis
5339        Jamario Moon
Name: Player, Length: 100, dtype: object


3. Clasificar el motivo de la no coincidencia

a) ¿El jugador existe en salarios pero en otra temporada?

In [114]:
salary_names = set(salary_old["PlayerKey"].unique())

missing_salary["ExistsInSalary"] = (
    missing_salary["PlayerKey"]
    .isin(salary_names)
)

print(
    missing_salary["ExistsInSalary"]
    .value_counts()
)

ExistsInSalary
False    304
True     159
Name: count, dtype: int64


Si ExistsInSalary = True, significa:

El jugador sí está en el dataset de salarios, pero no para esa temporada.

Eso suele indicar:

debut a mitad de temporada,
contratos especiales,
diferencias en el año de la temporada,
etc.

b) ¿Nunca aparece en el dataset de salarios?

Los que tengan:

ExistsInSalary = False

son probablemente:

contratos de 10 días;
jugadores que nunca llegaron a cobrar un salario registrado;
omisiones del dataset de Kaggle.

4. Buscar posibles coincidencias "casi iguales"

Podemos detectar si quedan nombres prácticamente iguales.

In [115]:
from difflib import get_close_matches

salary_names = salary_old["Player"].unique()

for player in missing_players.head(30):

    print(player)

    print(
        get_close_matches(
            player,
            salary_names,
            n=3,
            cutoff=0.85
        )
    )

    print("-"*40)

A.J. Guyton
['AJ Guyton']
----------------------------------------
Aaron Gray
['Aaron Gray']
----------------------------------------
Adonis Thomas
[]
----------------------------------------
Alan Anderson
['Alan Anderson', 'Alan Henderson']
----------------------------------------
Allen Crabbe
['Allen Crabbe']
----------------------------------------
Allonzo Trier
['Allonzo Trier']
----------------------------------------
Alonzo Gee
['Alonzo Gee']
----------------------------------------
Amile Jefferson
['Amile Jefferson', 'Al Jefferson']
----------------------------------------
Andre Barrett
['Andre Barrett']
----------------------------------------
Andy Panko
[]
----------------------------------------
Ansu Sesay
['Ansu Sesay']
----------------------------------------
Anthony Carter
['Anthony Carter', 'Anthony Parker', 'Anthony Barber']
----------------------------------------
Anthony Goldwire
['Anthony Goldwire']
----------------------------------------
Anthony Grundy
[]
----------

Tabla resumen

In [116]:
print(
    missing_salary
    .groupby("Season")["Player"]
    .nunique()
)

Season
2000-01    22
2001-02    23
2002-03    17
2003-04    31
2004-05    31
2005-06    22
2006-07    16
2007-08    18
2008-09    15
2009-10    18
2010-11    21
2011-12    35
2012-13    16
2013-14    24
2014-15    16
2015-16    16
2016-17    15
2017-18    16
2018-19    16
2019-20    41
2020-21    16
2021-22    18
Name: Player, dtype: int64


In [117]:
print(
    missing_salary["MP"]
    .describe()
)

count    463.000000
mean      15.463715
std        8.702531
min        1.000000
25%        8.950000
50%       14.000000
75%       20.900000
max       42.000000
Name: MP, dtype: float64


In [118]:
salary_old[
    salary_old["Player"] == "Aaron Gray"
]

,Player,Season,Salary,SalaryInflation,PlayerKey
3683,Aaron Gray,2007-08,427163.0,607496.0,aaron gray
4135,Aaron Gray,2008-09,711517.0,963509.0,aaron gray
4554,Aaron Gray,2009-10,1000497.0,1374445.0,aaron gray
4997,Aaron Gray,2010-11,1028840.0,1398649.0,aaron gray
5332,Aaron Gray,2011-12,2500000.0,3281813.0,aaron gray
6295,Aaron Gray,2013-14,2612500.0,3315200.0,aaron gray
6895,Aaron Gray,2014-15,1227985.0,1526646.0,aaron gray
7400,Aaron Gray,2015-16,1356146.0,1683893.0,aaron gray


In [119]:
nba_master_featured[
    nba_master_featured["Player"] == "Aaron Gray"
][["Player","Season","Team"]]

,Player,Season,Team
3434,Aaron Gray,2007-08,CHI
3914,Aaron Gray,2008-09,CHI
4374,Aaron Gray,2009-10,NOH
4832,Aaron Gray,2010-11,NOH
5259,Aaron Gray,2011-12,TOR
5784,Aaron Gray,2012-13,TOR
6302,Aaron Gray,2013-14,SAC


In [120]:
exists = missing_salary[
    missing_salary["ExistsInSalary"]
]

print(exists.shape)

exists[
    [
        "Player",
        "Season",
        "Team"
    ]
].sort_values(
    ["Player","Season"]
).head(100)

(159, 185)


,Player,Season,Team
1303,A.J. Guyton,2002-03,GSW
5784,Aaron Gray,2012-13,TOR
5079,Alan Anderson,2011-12,TOR
9244,Allen Crabbe,2019-20,ATL
9159,Allonzo Trier,2019-20,NYK
...,...,...,...
4731,Kyle Weaver,2010-11,UTA
5318,Kyrylo Fesenko,2011-12,IND
9291,Lance Thomas,2019-20,BRK
2055,Lawrence Funderburke,2004-05,CHI


In [121]:
print(f"Registros totales: {len(nba_master_featured)}")

print(f"Con salario: {test_merge['Salary'].notna().sum()}")

print(f"Sin salario: {test_merge['Salary'].isna().sum()}")

print(
    f"Cobertura: "
    f"{100 * test_merge['Salary'].notna().mean():.2f}%"
)

Registros totales: 12227
Con salario: 10084
Sin salario: 2143
Cobertura: 82.47%


# SALARIES_NEW

## 1. Cargamos el dataset

In [122]:
from pathlib import Path
import os

PROJECT_DIR = Path(r"C:\Users\marco\OneDrive\Desktop\Proyecto")
os.chdir(PROJECT_DIR)

print("Directorio de trabajo:", Path.cwd())

Directorio de trabajo: C:\Users\marco\OneDrive\Desktop\Proyecto


In [123]:
import pandas as pd
import numpy as np

In [124]:
salary_new = pd.read_csv(
    r"Datasets\Salaries - Kaggle\NBA Player Salaries_2000-2025.csv"
)

# 2. Auditoría inicial

In [125]:
print(salary_new.shape)

print(salary_new.columns)

print(salary_new.head())

print(salary_new.dtypes)

(12386, 3)
Index(['Player', 'Salary', 'Season'], dtype='object')
             Player    Salary  Season
0  Shaquille O'Neal  17142000    2000
1     Kevin Garnett  16806000    2000
2   Alonzo Mourning  15004000    2000
3      Juwan Howard  15000000    2000
4    Scottie Pippen  14795000    2000
Player    object
Salary     int64
Season     int64
dtype: object


# 3. Revisar temporadas

In [126]:
print(
    salary_new["Season"].min()
)

print(
    salary_new["Season"].max()
)

print(
    sorted(
        salary_new["Season"].unique()
    )[:5]
)

print(
    sorted(
        salary_new["Season"].unique()
    )[-5:]
)

2000
2025
[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004)]
[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


# 4. Revisar nombres

In [127]:
salary_new.sample(10)

,Player,Salary,Season
3304,Von Wafer,797581,2009
6453,John Henson,1987320,2015
4044,O.J. Mayo,4456200,2011
8829,Frank Mason III,1378242,2019
1299,Danny Fortson,5428000,2004
7527,Cameron Payne,2112480,2017
3080,Kendrick Perkins,4578880,2009
288,Eric Snow,3375000,2001
6069,Shelvin Mack,884293,2014
7539,Kevin Seraphin,1800000,2017


In [128]:
salary_new[
    salary_new["Player"]
    .str.contains("Jok", na=False)
]

,Player,Salary,Season
7024,Nikola Jokic,1300000,2016
7568,Nikola Jokic,1358500,2017
8188,Nikola Jokic,1471382,2018
8336,Peter Jok,815615,2018
8445,Nikola Jokic,24605181,2019
8952,Nikola Jokic,27504630,2020
9476,Nikola Jokic,36002595,2021
10059,Nikola Jokic,35898700,2022
10714,Nikola Jokic,35040189,2023
11259,Nikola Jokic,49021953,2024


In [129]:
salary_new[
    salary_new["Player"]
    .str.contains("Donc", na=False)
]

,Player,Salary,Season
8589,Luka Doncic,6560640,2019
9085,Luka Doncic,7683360,2020
9602,Luka Doncic,9809686,2021
10155,Luka Doncic,11765131,2022
10703,Luka Doncic,39332974,2023
11274,Luka Doncic,41254687,2024
11886,Luka Doncic,43031940,2025


In [130]:
salary_new[
    salary_new["Player"]
    .str.contains("Peja", na=False)
]

,Player,Salary,Season
404,Peja Stojakovic,1310000,2001
621,Peja Stojakovic,5000000,2002
1023,Peja Stojakovic,5625000,2003
1280,Peja Stojakovic,6250000,2004
1448,Peja Stojakovic,6875000,2005
1595,Peja Stojakovic,7525000,2006
2001,Peja Stojakovic,10800000,2007
2472,Peja Stojakovic,11664000,2008
2961,Peja Stojakovic,12528000,2009
3437,Peja Stojakovic,13392000,2010


# 5. Auditoría

In [131]:
print(salary_new.duplicated().sum())

print(
    salary_new[
        ["Player", "Season"]
    ].duplicated().sum()
)

0
0


In [132]:
print(
    salary_new.isna().sum()
)

Player    0
Salary    0
Season    0
dtype: int64


# 6. Crear la columna SeasonYear

In [133]:
salary_new = salary_new.rename(
    columns={
        "Season": "SeasonYear"
    }
)

# 7. Crear el formato de temporada

In [134]:
salary_new["Season"] = (
    salary_new["SeasonYear"].astype(str)
    + "-"
    + (
        (salary_new["SeasonYear"] + 1)
        .astype(str)
        .str[-2:]
    )
)

# 8. Crear PlayerKey

In [135]:
import re
import unicodedata
import numpy as np

def create_player_key(name):

    if pd.isna(name):
        return np.nan

    # Eliminar acentos
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    # Eliminar puntos y apóstrofes
    name = re.sub(r"[.'`´]", "", name)

    # Eliminar guiones
    name = name.replace("-", " ")

    # Eliminar sufijos
    name = re.sub(
        r"\b(JR|SR|II|III|IV)\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Espacios
    name = " ".join(name.split())

    return name.lower()

In [136]:
nba_master_featured = pd.read_csv(r"Datasets\Team Stats - Kaggle\nba_master_featured.csv")

In [137]:
nba_master_featured["PlayerKey"] = (
    nba_master_featured["Player"]
    .apply(create_player_key)
)

salary_new["PlayerKey"] = (
    salary_new["Player"]
    .apply(create_player_key)
)

# 9. Auditoría

In [138]:
print(salary_new.shape)

print(
    salary_new[["PlayerKey", "Season"]]
    .duplicated()
    .sum()
)

print(
    salary_new.isna().sum()
)

(12386, 5)
0
Player        0
Salary        0
SeasonYear    0
Season        0
PlayerKey     0
dtype: int64


# 10. Merge de prueba

In [139]:
test_merge2 = nba_master_featured.merge(
    salary_new,
    on=["PlayerKey", "Season"],
    how="left",
    indicator=True,
    suffixes=("", "_salary")
)

In [140]:
print(test_merge2["_merge"].value_counts())

print(
    test_merge2["Salary"]
    .isna()
    .sum()
)

_merge
both          8848
left_only     3379
right_only       0
Name: count, dtype: int64
3379


In [141]:
print(
    round(
        100
        * test_merge2["Salary"].notna().mean(),
        2
    )
)

72.36


# 11. Comparación

In [142]:
print(f"Registros totales: {len(nba_master_featured)}")

print(f"Con salario: {test_merge2['Salary'].notna().sum()}")

print(f"Sin salario: {test_merge2['Salary'].isna().sum()}")

print(
    f"Cobertura: "
    f"{100 * test_merge2['Salary'].notna().mean():.2f}%"
)

Registros totales: 12227
Con salario: 8848
Sin salario: 3379
Cobertura: 72.36%


In [143]:
print(salary_new["Season"].value_counts().sort_index())

Season
2000-01    157
2001-02    374
2002-03    416
2003-04    280
2004-05    180
2005-06    120
2006-07    436
2007-08    470
2008-09    490
2009-10    485
2010-11    481
2011-12    542
2012-13    619
2013-14    697
2014-15    434
2015-16    527
2016-17    521
2017-18    594
2018-19    598
2019-20    503
2020-21    528
2021-22    578
2022-23    653
2023-24    574
2024-25    612
2025-26    517
Name: count, dtype: int64


In [144]:
print(
    nba_master_featured["Season"]
    .value_counts()
    .sort_index()
)

Season
2000-01    441
2001-02    440
2002-03    428
2003-04    442
2004-05    464
2005-06    458
2006-07    458
2007-08    450
2008-09    445
2009-10    442
2010-11    452
2011-12    478
2012-13    469
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
Name: count, dtype: int64


In [145]:
master_players = set(nba_master_featured["PlayerKey"].unique())
salary_players = set(salary_new["PlayerKey"].unique())

print("Jugadores master:", len(master_players))
print("Jugadores salary:", len(salary_players))

print("Coinciden:", len(master_players & salary_players))
print("Solo master:", len(master_players - salary_players))
print("Solo salary:", len(salary_players - master_players))

Jugadores master: 2465
Jugadores salary: 2378
Coinciden: 2197
Solo master: 268
Solo salary: 181


### En este dataset nos faltan más datos que en el anterior, sobre todo en las primeras temporadas. Por lo tanto, nos quedaremos con salaries_old para 2000-01 a 2021-22 y salaries_new 2022-23 a 2024-25.

## 1. Comprobar que ambos datasets tienen las mismas columnas

In [148]:
salary_old["SeasonYear"] = (
    salary_old["Season"]
    .str[:4]
    .astype(int)
)

salary_old

In [152]:
print(salary_old.columns)

Index(['Player', 'Season', 'Salary', 'PlayerKey', 'SeasonYear'], dtype='object')


salary_new

In [150]:
print(salary_new.columns)

Index(['Player', 'Salary', 'SeasonYear', 'Season', 'PlayerKey'], dtype='object')


## 2. Eliminar la columna de inflación del dataset antiguo

In [151]:
salary_old = salary_old.drop(
    columns=["SalaryInflation"],
    errors="ignore"
)

## 3. Filtrar únicamente las temporadas históricas

In [153]:
salary_old = salary_old[
    salary_old["Season"] <= "2021-22"
].copy()

Auditoría:

In [154]:
print(
    salary_old["Season"].min(),
    salary_old["Season"].max()
)

2000-01 2021-22


## 4. Filtrar las temporadas recientes

In [155]:
salary_recent = salary_new[
    salary_new["Season"].isin([
        "2022-23",
        "2023-24",
        "2024-25"
    ])
].copy()

Auditoría:

In [156]:
print(
    salary_recent["Season"].value_counts()
)

Season
2022-23    653
2024-25    612
2023-24    574
Name: count, dtype: int64


## 5. Seleccionar únicamente las columnas necesarias

In [157]:
columns = [
    "Player",
    "PlayerKey",
    "Season",
    "SeasonYear",
    "Salary"
]

In [158]:
salary_old = salary_old[columns]

salary_recent = salary_recent[columns]

## 6. Concatenar

In [159]:
salary_final = pd.concat(
    [
        salary_old,
        salary_recent
    ],
    ignore_index=True
)

## 7. Auditoría completa

Dimensiones

In [160]:
print(salary_final.shape)

(12851, 5)


Duplicados

In [161]:
print(
    salary_final.duplicated().sum()
)

0


Duplicados por clave

In [162]:
print(
    salary_final[
        ["PlayerKey", "Season"]
    ].duplicated().sum()
)

0


Temporadas

In [163]:
print(
    sorted(
        salary_final["Season"].unique()
    )
)

['2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


Nulos

In [164]:
print(
    salary_final.isna().sum()
)

Player        0
PlayerKey     0
Season        0
SeasonYear    0
Salary        0
dtype: int64


## 8. Guardar el dataset salarial definitivo

In [165]:
salary_final.to_parquet(
    r"Datasets\Salaries - Kaggle\salary_final.parquet",
    index=False
)

salary_final.to_csv(
    r"Datasets\Salaries - Kaggle\salary_final.csv",
    index=False,
    encoding="utf-8-sig"
)

## 9. Merge definitivo

In [166]:
nba_master_final = nba_master_featured.merge(
    salary_final[
        [
            "PlayerKey",
            "Season",
            "Salary"
        ]
    ],
    on=["PlayerKey", "Season"],
    how="left", indicator=True
)

## 10. Eliminar la clave auxiliar

In [167]:
nba_master_final = nba_master_final.drop(
    columns="PlayerKey"
)

## 11. Auditoría final

In [168]:
print("="*60)
print("MASTER FINAL")

print(nba_master_final.shape)

print(
    nba_master_final[
        ["Player","Season","Team"]
    ].duplicated().sum()
)

print(
    nba_master_final["Salary"]
    .isna().sum()
)

print(
    f"Cobertura salarial: "
    f"{100*nba_master_final['Salary'].notna().mean():.2f}%"
)

MASTER FINAL
(12227, 181)
0
812
Cobertura salarial: 93.36%


In [169]:
missing_salary = nba_master_final[
    nba_master_final["Salary"].isna()
].copy()

print(missing_salary.shape)

print(
    missing_salary["Season"]
    .value_counts()
    .sort_index()
)

(812, 181)
Season
2000-01     22
2001-02     23
2002-03     17
2003-04     31
2004-05     31
2005-06     22
2006-07     16
2007-08     18
2008-09     15
2009-10     18
2010-11     21
2011-12     35
2012-13     16
2013-14     24
2014-15     16
2015-16     16
2016-17     15
2017-18     16
2018-19     16
2019-20     41
2020-21     16
2021-22     18
2022-23    103
2023-24    125
2024-25    121
Name: count, dtype: int64


## 12. Guardar el dataset definitivo

In [170]:
nba_master_final.to_parquet(
    r"Datasets\nba_master_final.parquet",
    index=False
)

nba_master_final.to_csv(
    r"Datasets\nba_master_final.csv",
    index=False,
    encoding="utf-8-sig"
)